In [2]:
# ============================================================
# Supplementary Figure 4
# NMF rank selection:
# Reconstruction error + relative error reduction
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import NMF

# ============================================================
# 1. Path config
# ============================================================

PROJECT_DIR = "/mnt/zhangzheng_group/liuz-53/Test_R/R_loop_HCC"
FIG2_DIR = os.path.join(PROJECT_DIR, "02_Figure2_python")
SUPP_DIR = os.path.join(PROJECT_DIR, "02_Figure2_python")

os.makedirs(SUPP_DIR, exist_ok=True)

PSEUDOBULK_FILE = os.path.join(
    FIG2_DIR,
    "Fig2_pseudobulk_Rloop_expression.csv"
)

OUT_PREFIX = os.path.join(
    SUPP_DIR,
    "SupplementaryFig4_NMF_rank_selection"
)

# ============================================================
# 2. Plot setting
# ============================================================

mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["axes.unicode_minus"] = False

# ============================================================
# 3. Load pseudo-bulk R-loop matrix
# ============================================================

pseudo_df = pd.read_csv(PSEUDOBULK_FILE)

non_gene_cols = ["pseudo_group", "n_cells"]
gene_cols = [c for c in pseudo_df.columns if c not in non_gene_cols]

gene_mat = pseudo_df[gene_cols].copy()

# remove genes with zero variance
gene_mat = gene_mat.loc[:, gene_mat.var(axis=0) > 0]

print("Pseudo-bulk matrix used for NMF:")
print(gene_mat.shape)

# Same scaling as main Figure 3
X_nmf = MinMaxScaler().fit_transform(gene_mat.values)

# ============================================================
# 4. Run NMF across ranks and random seeds
# ============================================================

rank_list = list(range(2, 9))      # k = 2–8
seed_list = list(range(1, 21))     # 20 repeated runs

records = []

for k in rank_list:
    print(f"Running NMF rank k = {k}")

    for seed in seed_list:
        nmf = NMF(
            n_components=k,
            init="nndsvda",
            random_state=seed,
            max_iter=1000,
            solver="cd"
        )

        W = nmf.fit_transform(X_nmf)
        H = nmf.components_

        records.append({
            "rank": k,
            "seed": seed,
            "reconstruction_error": nmf.reconstruction_err_
        })

result_df = pd.DataFrame(records)

result_df.to_csv(
    os.path.join(SUPP_DIR, "SupplementaryFig4_NMF_rank_selection_metrics.csv"),
    index=False
)

# ============================================================
# 5. Summary table
# ============================================================

summary_df = (
    result_df
    .groupby("rank")
    .agg(
        reconstruction_error_mean=("reconstruction_error", "mean"),
        reconstruction_error_sd=("reconstruction_error", "std")
    )
    .reset_index()
)

# Relative reduction in reconstruction error
summary_df["relative_reduction_percent"] = np.nan

for i in range(1, summary_df.shape[0]):
    previous_error = summary_df.loc[i - 1, "reconstruction_error_mean"]
    current_error = summary_df.loc[i, "reconstruction_error_mean"]

    reduction = (previous_error - current_error) / previous_error * 100
    summary_df.loc[i, "relative_reduction_percent"] = reduction

summary_df.to_csv(
    os.path.join(SUPP_DIR, "SupplementaryFig4_NMF_rank_selection_summary.csv"),
    index=False
)

print(summary_df)

# ============================================================
# 6. Plot Supplementary Fig. 4
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.6))

# ----------------------------
# Fig. S4A: Reconstruction error
# ----------------------------
ax = axes[0]

ax.errorbar(
    summary_df["rank"],
    summary_df["reconstruction_error_mean"],
    yerr=summary_df["reconstruction_error_sd"],
    marker="o",
    linewidth=1.6,
    capsize=3
)

ax.axvline(
    x=4,
    linestyle="--",
    linewidth=1
)

ax.set_xlabel("NMF rank (k)")
ax.set_ylabel("Reconstruction error")
ax.set_title("Reconstruction error")
ax.set_xticks(rank_list)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.text(
    4.1,
    summary_df["reconstruction_error_mean"].max() * 0.96,
    "Selected k = 4",
    fontsize=9,
    va="top"
)

# ----------------------------
# Fig. S4B: Relative reduction
# ----------------------------
ax = axes[1]

plot_df = summary_df.dropna(subset=["relative_reduction_percent"])

ax.plot(
    plot_df["rank"],
    plot_df["relative_reduction_percent"],
    marker="o",
    linewidth=1.6
)

ax.axvline(
    x=4,
    linestyle="--",
    linewidth=1
)

ax.set_xlabel("NMF rank (k)")
ax.set_ylabel("Relative reduction in error (%)")
ax.set_title("Relative error reduction")
ax.set_xticks(rank_list)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.text(
    4.1,
    plot_df["relative_reduction_percent"].max() * 0.95,
    "Selected k = 4",
    fontsize=9,
    va="top"
)

# Panel labels
axes[0].text(
    -0.18,
    1.08,
    "A",
    transform=axes[0].transAxes,
    fontsize=16,
    fontweight="bold",
    va="top"
)

axes[1].text(
    -0.18,
    1.08,
    "B",
    transform=axes[1].transAxes,
    fontsize=16,
    fontweight="bold",
    va="top"
)

plt.tight_layout()

# ============================================================
# 7. Save
# ============================================================

for ext in ["pdf", "png", "svg"]:
    fig.savefig(
        f"{OUT_PREFIX}.{ext}",
        dpi=300,
        bbox_inches="tight",
        transparent=True
    )

plt.close(fig)

print("Done.")
print("Supplementary Fig. 4 saved to:")
print(SUPP_DIR)

Pseudo-bulk matrix used for NMF:
(239, 1095)
Running NMF rank k = 2
Running NMF rank k = 3
Running NMF rank k = 4
Running NMF rank k = 5
Running NMF rank k = 6
Running NMF rank k = 7
Running NMF rank k = 8
   rank  reconstruction_error_mean  reconstruction_error_sd  \
0     2                  77.917672             1.694046e-14   
1     3                  73.793437             4.207964e-12   
2     4                  70.802189             8.770945e-11   
3     5                  68.621708             3.414164e-09   
4     6                  66.454598             1.884565e-09   
5     7                  64.811926             4.438996e-08   
6     8                  63.648993             4.122266e-07   

   relative_reduction_percent  
0                         NaN  
1                    5.293067  
2                    4.053542  
3                    3.079680  
4                    3.158053  
5                    2.471872  
6                    1.794319  
Done.
Supplementary Fig. 4 saved 